# TP-02: Cleaning Data - Applying OOP and Design Patterns
## Exercise 1: Build the CSVReader Class (OOP Foundation)

In [1]:
import os
import pandas as pd

### Instruction
1.	Create a class CSVReader.
2.	Define an `__init__` method that takes file_path.
3.	Add a read() method that loads the CSV file using pandas.
4.	Add a preview(n) method that prints the top n rows.

In [2]:
class CSVReader:
    
    def __init__(self, file_path):
        self.file_path = file_path
        self._data = None  

    def read(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found: {self.file_path}")
        self._data = pd.read_csv(self.file_path)
        return self._data

    def preview(self, n=5):
        if self._data is None:
            self.read()
        print(f"Top {n} rows of '{self.file_path}':")
        print(self._data.head(n))

    @property
    def data(self):
        return self._data

    def __repr__(self):
        status = "loaded" if self._data is not None else "not loaded"
        return f"CSVReader(file_path='{self.file_path}', status={status})"

In [3]:
# Test the class with sample_data.csv
reader = CSVReader("sample_data.csv")
print(reader)

CSVReader(file_path='sample_data.csv', status=not loaded)


In [4]:
df = reader.read()
print(reader)
print("Shape:", df.shape)
df

CSVReader(file_path='sample_data.csv', status=loaded)
Shape: (7, 7)


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [5]:
reader.preview(3)

Top 3 rows of 'sample_data.csv':
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN


In [6]:
print(type(reader.data))
print("Missing values per column:")
print(reader.data.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
Missing values per column:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64


In [7]:
try:
    CSVReader("not_exist.csv").read()
except FileNotFoundError as e:
    print("Error:", e)

Error: File not found: not_exist.csv


## Exercise 2: Apply Pattern for Missing Values Handling

In [8]:
import numpy as np
from abc import ABC, abstractmethod
from typing import Protocol

### Instruction
1.	Create an abstract class MissingValueStrategy with a method handle(df).
2.	Create subclasses:
    - DropMissing
    - FillMean
    - FillMode
3.	Create a DataCleaner class that applies the chosen strategy.

In [10]:
# Load the dataset
df = pd.read_csv('sample_data.csv')

print("Original Dataset:")
print(df)
print("\nMissing Values Count:")
print(df.isnull().sum())
print("\nData Types:")
print(df.dtypes)

Original Dataset:
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles    NaN
6   7    Grace   NaN      162.0       60.0      Chicago   92.0

Missing Values Count:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64

Data Types:
id             int64
name          object
age          float64
height_cm    float64
weight_kg    float64
city          object
score        float64
dtype: object


In [11]:
# Abstract Base Class: MissingValueStrategy
class MissingValueStrategy(ABC):
    """Abstract base class for missing value handling strategies."""
    
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Handle missing values in the dataframe.
        
        Args:
            df: Input dataframe with missing values
            
        Returns:
            DataFrame with missing values handled
        """
        pass
    
    def __str__(self):
        return self.__class__.__name__

In [12]:
# Strategy 1: DropMissing
class DropMissing(MissingValueStrategy):
    """Strategy to drop rows with any missing values."""
    
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Drop all rows containing missing values."""
        result = df.dropna()
        print(f"Dropped {len(df) - len(result)} rows with missing values.")
        return result

In [13]:
# Strategy 2: FillMean
class FillMean(MissingValueStrategy):
    """Strategy to fill missing values with the mean of numeric columns."""
    
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fill missing values with column means for numeric columns."""
        result = df.copy()
        
        # Get numeric columns only
        numeric_cols = result.select_dtypes(include=[np.number]).columns
        
        for col in numeric_cols:
            if result[col].isnull().any():
                mean_value = result[col].mean()
                result[col].fillna(mean_value, inplace=True)
                print(f"Filled {col} with mean: {mean_value:.2f}")
        
        # For non-numeric columns, we could drop rows or use a different strategy
        # Here we'll just leave them as is
        return result

In [14]:
# Strategy 3: FillMode
class FillMode(MissingValueStrategy):
    """Strategy to fill missing values with the mode (most frequent value)."""
    
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fill missing values with column mode."""
        result = df.copy()
        
        for col in result.columns:
            if result[col].isnull().any():
                mode_value = result[col].mode()
                if len(mode_value) > 0:
                    result[col].fillna(mode_value[0], inplace=True)
                    print(f"Filled {col} with mode: {mode_value[0]}")
        
        return result

In [15]:
class DataCleaner:
    """Context class that applies a missing value handling strategy."""
    
    def __init__(self, strategy: MissingValueStrategy):
        """Initialize with a specific strategy.
        
        Args:
            strategy: A MissingValueStrategy instance
        """
        self._strategy = strategy
    
    def set_strategy(self, strategy: MissingValueStrategy):
        """Change the strategy at runtime.
        
        Args:
            strategy: A new MissingValueStrategy instance
        """
        self._strategy = strategy
        print(f"Strategy changed to: {strategy}")
    
    def clean_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """Clean the dataframe using the current strategy.
        
        Args:
            df: Input dataframe
            
        Returns:
            Cleaned dataframe
        """
        print(f"\nApplying strategy: {self._strategy}")
        print("-" * 50)
        return self._strategy.handle(df)

In [16]:
# Test 1: DropMissing
# Create a cleaner with DropMissing strategy
cleaner = DataCleaner(DropMissing())
df_dropped = cleaner.clean_data(df)

print("\nResult after dropping missing values:")
print(df_dropped)
print(f"\nShape: {df_dropped.shape}")


Applying strategy: DropMissing
--------------------------------------------------
Dropped 5 rows with missing values.

Result after dropping missing values:
   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
4   5    Eva  27.0      160.0       55.0  New York   88.0

Shape: (2, 7)


In [17]:
# Test 2: FillMean
# Change strategy to FillMean
cleaner.set_strategy(FillMean())
df_mean_filled = cleaner.clean_data(df)

print("\nResult after filling with mean:")
print(df_mean_filled)
print("\nMissing values after filling with mean:")
print(df_mean_filled.isnull().sum())

Strategy changed to: FillMean

Applying strategy: FillMean
--------------------------------------------------
Filled age with mean: 30.25
Filled height_cm with mean: 167.00
Filled weight_kg with mean: 70.00
Filled score with mean: 86.00

Result after filling with mean:
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0

Missing values after filling with mean:
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64


/var/folders/qb/8q46b91x0jvbcmpx5sk7229c0000gn/T/ipykernel_18872/494901066.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  result[col].fillna(mean_value, inplace=True)


In [18]:
# Test 3: FillMode
# Change strategy to FillMode
cleaner.set_strategy(FillMode())
df_mode_filled = cleaner.clean_data(df)

print("\nResult after filling with mode:")
print(df_mode_filled)
print("\nMissing values after filling with mode:")
print(df_mode_filled.isnull().sum())

Strategy changed to: FillMode

Applying strategy: FillMode
--------------------------------------------------
Filled age with mode: 27.0
Filled height_cm with mode: 160.0
Filled weight_kg with mode: 55.0
Filled score with mode: 75.0

Result after filling with mode:
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob  27.0      172.0       55.0  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago   75.0
3   4    David  27.0      160.0       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles   75.0
6   7    Grace  27.0      162.0       60.0      Chicago   92.0

Missing values after filling with mode:
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64


/var/folders/qb/8q46b91x0jvbcmpx5sk7229c0000gn/T/ipykernel_18872/3372783794.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  result[col].fillna(mode_value[0], inplace=True)


## Exercise 3: Add Decorators for Logging and Timing

In [19]:
import time
from functools import wraps

### Instructions
1.	Create two decorators:
    - `@log_action` → logs when a method starts and finishes.
    - `@log_time` → calculates and prints execution time.
2.	Apply them to methods in CSVReader or DataCleaner.


In [20]:
# 1. Decorator for logging
def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[START] {func.__name__}")
        result = func(*args, **kwargs)
        print(f"[FINISH] {func.__name__}")
        return result
    return wrapper

In [21]:
# 2. Decorator for execution time
def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()

        print(f"[TIME] {func.__name__}: {end - start:.6f} seconds")
        return result
    return wrapper

In [22]:
# 3. CSVReader class from Exercise 1
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None

    @log_action
    @log_time
    def read(self):
        self.df = pd.read_csv(self.file_path)
        return self.df

    @log_action
    def preview(self, n=5):
        if self.df is None:
            self.read()
        print(self.df.head(n))

In [23]:
# 4. Test with sample CSV
reader = CSVReader("sample_data.csv")

df = reader.read()
reader.preview(5)

[START] read
[TIME] read: 0.005598 seconds
[FINISH] read
[START] preview
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
[FINISH] preview


## Exercise 4: Implement Factory Pattern for Data Transformations

### Instructions:
1.	Create an abstract class DataTransform with a method apply(df).
2.	Implement subclasses:
    - NormalizeColumns
    - RemoveDuplicates
    - StandardizeText
3.	Create a TransformFactory that returns transformation objects based on string input.


In [24]:
# 1. Abstract Class
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

In [25]:
# 2. Subclasses
class NormalizeColumns(DataTransform):
    """Makes column names lowercase and replaces spaces with underscores."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
        return df

In [26]:
class RemoveDuplicates(DataTransform):
    """Removes duplicate rows from the DataFrame."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().reset_index(drop=True)

In [27]:
class StandardizeText(DataTransform):
    """Trims extra spaces and converts text values to lowercase."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        text_cols = df.select_dtypes(include=["object", "string"]).columns
        for col in text_cols:
            df[col] = df[col].astype(str).str.strip().str.lower()
        return df

In [28]:
# 3. Factory Class
class TransformFactory:
    @staticmethod
    def create_transform(transform_type: str) -> DataTransform:
        transforms = {
            "normalize_columns": NormalizeColumns,
            "remove_duplicates": RemoveDuplicates,
            "standardize_text": StandardizeText,
        }
        key = transform_type.strip().lower()
        if key not in transforms:
            raise ValueError(f"Unknown transformation: '{transform_type}'")
        return transforms[key]()

In [30]:
df = pd.read_csv("sample_data.csv")

# Apply transformations using string names
pipeline = ["normalize_columns", "standardize_text", "remove_duplicates"]

for step in pipeline:
    transformer = TransformFactory.create_transform(step)
    df = transformer.apply(df)

print(df)

   id     name   age  height_cm  weight_kg         city  score
0   1    alice  29.0      165.0       68.0     new york   85.0
1   2      bob   NaN      172.0        NaN  los angeles   90.0
2   3  charlie  35.0      168.0       72.0      chicago    NaN
3   4    david   NaN        NaN       80.0      houston   75.0
4   5      eva  27.0      160.0       55.0     new york   88.0
5   6    frank  30.0      175.0       85.0  los angeles    NaN
6   7    grace   NaN      162.0       60.0      chicago   92.0


## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

In [31]:
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

### Instructions:
1.	Create an abstract class DataPipeline with a `run()` method defining steps:
    - `load()`
    - `clean()`
    - `transform()`
    - `save()`
2.	Create `CSVDataPipeline` that implements each step using previous classes.

In [32]:
# Boilerplate Code Template Implementation for entire pipeline
class DataPipeline(ABC):

    def __init__(self, source: Path, destination: Path) -> None:
        self.source = source
        self.destination = destination
        self.df: pd.DataFrame | None = None

    @abstractmethod
    def load(self) -> None: ...

    @abstractmethod
    def clean(self) -> None: ...

    @abstractmethod
    def transform(self) -> None: ...

    @abstractmethod
    def save(self) -> None: ...

    @log_action
    @log_time
    def run(self) -> None:
        LOGGER.info("PIPELINE START: %s to %s", self.source, self.destination)
        self.load()
        self.clean()
        self.transform()
        self.save()
        LOGGER.info("PIPELINE END: result saved in %s", self.destination)

In [33]:
# Subclass core logics rule abstract from DataPipeline
class CSVDataPipeline(DataPipeline):

    def __init__(self, source: Path, destination: Path,
                 cleaner: DataCleaner, transforms: list[DataTransform]) -> None:
        super().__init__(source, destination)
        self.reader = CSVReader(source)
        self.cleaner = cleaner
        self.transforms = list(transforms)

    def load(self) -> None:
        self.df = self.reader.read()
        self.reader.preview(3)

    def clean(self) -> None:
        self.df = self.cleaner.clean(self.df)

    def transform(self) -> None:
        for transform in self.transforms:
            LOGGER.info("Transform: %s", type(transform).__name__)
            self.df = transform.apply(self.df)

    def save(self) -> None:
        self.df.to_csv(self.destination, index=False)

In [38]:
def build_pipeline(strategy: MissingValueStrategy, transform_names: list[str]) -> CSVDataPipeline:
    return CSVDataPipeline(
        source="/Users/somchannreaksmey/repo-adv/sample_data.csv",
        destination="/Users/somchannreaksmey/repo-adv/",
        cleaner=DataCleaner(strategy),
        transforms=[TransformFactory.create_transform(name) for name in transform_names],
    )

In [43]:
import logging

pipeline = build_pipeline(
    strategy=FillMean(),
    transform_names=[
        "remove_duplicates",
        "standardize_text",
        "normalize_columns",
    ],
)

logging.basicConfig(level=logging.INFO)
LOGGER = logging.getLogger("data_pipeline")

LOGGER.info("[INFO]: Collaborators of the pipeline:")
LOGGER.info("  reader     : %s", type(pipeline.reader).__name__)
LOGGER.info(
    "  cleaner    : %s using %s",
    type(pipeline.cleaner).__name__,
    type(pipeline.cleaner._strategy).__name__,
)
LOGGER.info("  transforms : %s",
            [type(t).__name__ for t in pipeline.transforms])

pipeline.cleaner.clean = pipeline.cleaner.clean_data
pipeline.destination = Path("cleaned_sample_data.csv")

pipeline.run()

INFO:data_pipeline:[INFO]: Collaborators of the pipeline:
INFO:data_pipeline:  reader     : CSVReader
INFO:data_pipeline:  cleaner    : DataCleaner using FillMean
INFO:data_pipeline:  transforms : ['RemoveDuplicates', 'StandardizeText', 'NormalizeColumns']
INFO:data_pipeline:PIPELINE START: /Users/somchannreaksmey/repo-adv/sample_data.csv to cleaned_sample_data.csv
/var/folders/qb/8q46b91x0jvbcmpx5sk7229c0000gn/T/ipykernel_18872/494901066.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  result[col].fillna(mean_value, inpla

[START] run
[START] read
[TIME] read: 0.001668 seconds
[FINISH] read
[START] preview
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
[FINISH] preview

Applying strategy: FillMean
--------------------------------------------------
Filled age with mean: 30.25
Filled height_cm with mean: 167.00
Filled weight_kg with mean: 70.00
Filled score with mean: 86.00
[TIME] run: 0.012040 seconds
[FINISH] run
